In [1]:
import oursin as urchin
import pandas as pd
import numpy as np
from brainglobe_atlasapi import BrainGlobeAtlas

In [ ]:
urchin.setup(id='me', localhost=True)

(URN) connected to server
Login sent with ID: me, copy this ID into the renderer to connect.


(Camera receive) Camera CameraMain received an image
(Camera receive) Camera CameraMain received an image
(Camera receive) Camera CameraMain received an image
(Camera receive) Camera CameraMain received an image
(Camera receive) Camera CameraMain received an image
(Camera receive) Camera CameraMain received an image
(Camera receive) Camera CameraMain received an image
(Camera receive) Camera CameraMain received an image
(Camera receive) Camera CameraMain received an image


In [20]:
urchin.ccf25.load()

(Warning) Atlas was already loaded, the renderer can have issues if you try to load an atlas twice.


In [21]:


defaults = ['FRP','MO','SS','GU','VISC','AUD','VIS','ACA','PL',
            'ILA','ORB','AI','RSP','PTLp','TEa','PERI','OLF',
            'ECT','HPF','CTXsp','CNU','BS','CB']
area_list = urchin.ccf25.get_areas(defaults)

urchin.ccf25.set_visibilities(area_list, True)
urchin.ccf25.set_materials(area_list, 'transparent-unlit')
urchin.ccf25.set_alphas(area_list, 0.15)

In [22]:
urchin.camera.main.set_background_color("#000000")
urchin.camera.main.set_rotation('angled')
urchin.camera.main.set_mode("perspective")
urchin.camera.main.set_zoom(105)

In [7]:
# Load the file data/uuid_mlapdv.csv

raw = pd.read_csv('./data/raw/uuid_mlapdv.csv')

In [8]:
raw.head()

,uuid,ml,ap,dv
0,c7d051dc-1358-422d-9eab-ca13c6b6be1b,3450.0,7937.0,3927.0
1,38bc27f2-7301-405a-a870-54e0a7b06a26,3450.0,7933.0,3908.0
2,1c4e5e11-ca9d-441a-bb8f-4b56dcc95054,3450.0,7928.0,3888.0
3,b48a90ed-84f4-424d-87ae-92c22a423777,3450.0,7928.0,3888.0
4,38a812b9-b374-46f8-ab14-3dd146411fc8,3450.0,7928.0,3888.0


In [301]:
# print the range of each of the ml ap and dv columns in raw
print(raw[['ml','ap','dv']].min())
print(raw[['ml','ap','dv']].max())

ml     860.0
ap    1804.0
dv       1.0
dtype: float64
ml     9207.0
ap    13125.0
dv     7510.0
dtype: float64


In [9]:
atlas = BrainGlobeAtlas("allen_mouse_25um")

In [303]:
atlas.annotation.shape

(528, 320, 456)

In [10]:
# Get the colors
struct_df = pd.read_csv('./structures.csv')



def get_color(apdvlr):
    # re-order to ap/dv/ml
    ap = apdvlr[0]/25
    dv = apdvlr[1]/25
    ml = apdvlr[2]/25

    ap = np.clip(ap, 0, atlas.annotation.shape[0]-1)
    dv = np.clip(dv, 0, atlas.annotation.shape[1]-1)
    ml = np.clip(ml, 0, atlas.annotation.shape[2]-1)

    id = atlas.annotation[int(ap), int(dv), int(ml)]

    if not id or id == 0:
        return '#ffffff'

    color = struct_df[struct_df['id'] == id]['rgb_triplet'].values[0]

    # color is a string list of int [r, g, b], convert to list of ints
    color = color[1:-1].split(', ')
    color = [int(c) for c in color]

    # convert to hex
    color = '#%02x%02x%02x' % tuple(color)
    return color

In [ ]:
# Use the get_color function to get the colors from the mlapdv coordinates

raw_colors = []
for i, row in raw.iterrows():
    apdvlr = np.array(row[['ap','dv','ml']])
    raw_colors.append(get_color(apdvlr))

raw = raw.assign(color=raw_colors)

In [19]:

# Remove rows where color is '#ffffff'
raw = raw[raw['color'] != '#ffffff']

In [12]:
# Load the clu_avgs_dict.npy file
clu_avgs_dict = np.load('./data/raw/clu_avgs_dict.npy', allow_pickle=True).item()

In [307]:
np.average(clu_avgs_dict["000205d2-ce3c-443e-978a-f6b5307e80e5"][0][0])

5.84352207579389

In [23]:
pmeshes = urchin.meshes.create(len(raw), interactive=False) #creates 2 primitives, stored in list pmeshes

In [14]:
def spread_skewed_values(values, target_min=0, target_max=0.05):
    """Transform skewed values (0 to 250) into a more evenly spread range (0 to 0.05)."""
    values = np.array(values)
    transformed = np.sqrt(values)  # Apply square root transformation
    normalized = (transformed - transformed.min()) / (transformed.max() - transformed.min())
    return target_min + normalized * (target_max - target_min)


In [24]:
# reorder to AP/ML/DV for Urchin and make a list of lists

def rn():
  return np.random.rand()*100-50

mode_black = False
cheat = False

coords_list = []
colors_list = []
avgs_list = []
for i, row in raw.iterrows():
  ml = row['ml']
  if cheat and rn() < 0:
    ml = 11400 - row['ml']
  coords_list.append([row['ap']+rn(), ml+rn(), row['dv']+rn()])
  colors_list.append(row['color'])
  # also get the average firing rate
  avg = np.average(clu_avgs_dict[row['uuid']][0][0,100])
  avgs_list.append(avg)


if mode_black:
  colors_list = [[10, 10, 10]]*len(raw)

avgs_list = np.array(avgs_list)
avgs_list[np.isnan(avgs_list)] = 0
sizes_list = spread_skewed_values(avgs_list, target_min=0.01, target_max=0.07)

sizes_list = [[x, x, x] for x in sizes_list]

urchin.meshes.set_positions(pmeshes,coords_list) #sets the positions of the primitives
urchin.meshes.set_colors(pmeshes, colors_list)
urchin.meshes.set_scales(pmeshes, list(sizes_list))

In [25]:
urchin.camera.main.set_background_color("#000000")
urchin.camera.main.set_rotation('angled')
urchin.camera.main.set_mode("perspective")
urchin.camera.main.set_zoom(50)

In [37]:
urchin.camera.main.set_background_color("#000000")
urchin.camera.main.set_rotation('axial')
urchin.camera.main.set_mode("perspective")
urchin.camera.main.set_zoom(50)

In [341]:
# mouse rotation
urchin.camera.main.set_background_color("#000000")
urchin.camera.main.set_rotation([28,28,225])
urchin.camera.main.set_mode("perspective")
urchin.camera.main.set_zoom(105)

In [38]:

await urchin.camera.main.screenshot(size=[2560*2, 2560*2], filename='./axial.png')

(Camera receive) CameraMain complete


In [ ]:
# Animation
